# Sales Analytics & Demand Forecasting

End-to-end dairy sales analysis using Python, EDA and regression models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# Generate the dataset first by running data/generate_dataset.py
df = pd.read_csv('../data/dairy_sales_50000.csv', parse_dates=['Date'])
df.head()

In [ ]:
df.info()
df.describe().T

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
monthly = df.groupby(df['Date'].dt.to_period('M')).agg(Revenue=('Revenue','sum'), Units_Sold=('Units_Sold','sum'), Profit=('Profit','sum')).reset_index()
monthly['Date'] = monthly['Date'].dt.to_timestamp()
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(monthly['Date'], monthly['Revenue'])
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
plt.tight_layout()
plt.show()

In [ ]:
product_summary = df.groupby(['Product','Category']).agg(Units_Sold=('Units_Sold','sum'), Revenue=('Revenue','sum'), Profit=('Profit','sum')).sort_values('Revenue', ascending=False)
product_summary.head(10)

In [ ]:
region_summary = df.groupby('Region').agg(Revenue=('Revenue','sum'), Profit=('Profit','sum'), Units_Sold=('Units_Sold','sum')).sort_values('Revenue', ascending=False)
region_summary

In [ ]:
channel_summary = df.groupby('Channel').agg(Revenue=('Revenue','sum'), Profit=('Profit','sum'), Units_Sold=('Units_Sold','sum')).sort_values('Revenue', ascending=False)
channel_summary

## Demand Forecasting

Predict Units Sold from calendar, product, geography, channel, price and discount features.

In [ ]:
model_df = df.copy()
model_df['Year'] = model_df['Date'].dt.year
model_df['Month'] = model_df['Date'].dt.month
model_df['DayOfWeek'] = model_df['Date'].dt.dayofweek
features = ['Year','Month','DayOfWeek','Product','Category','Region','Channel','Customer_Type','Unit_Price','Discount_Pct']
X = model_df[features]
y = model_df['Units_Sold']
cat_cols = ['Product','Category','Region','Channel','Customer_Type']
num_cols = ['Year','Month','DayOfWeek','Unit_Price','Discount_Pct']
preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols), ('num','passthrough',num_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
models = {'Linear Regression': LinearRegression(), 'Decision Tree': DecisionTreeRegressor(max_depth=12, random_state=42)}
results = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({'Model': name, 'MAE': mean_absolute_error(y_test,pred), 'RMSE': mean_squared_error(y_test,pred)**0.5, 'R2': r2_score(y_test,pred)})
pd.DataFrame(results).sort_values('RMSE')

## Business Interpretation

Use product, regional and channel summaries to prioritize inventory and identify high-value segments. Compare MAE, RMSE and R² before selecting the baseline forecasting model.